# Fine-tune Qwen3-8B (QLoRA) on receptionist conversations

Free-stack training: Google Colab **T4 GPU** runtime + Unsloth + the JSONL exported by
`scripts/export_finetune.py` (TRL chat format, policy-stamped, PII-masked).

**Before promoting the result** to production, run `scripts/eval_battery.py` against the
fine-tuned model — it must match Gemini's 0-FAIL. See `TRAINING.md`.

Runtime: Runtime → Change runtime type → **T4 GPU**.

In [ ]:
%pip install -q unsloth  # pulls torch/transformers/trl/peft/bitsandbytes compatible with Colab T4

In [ ]:
from unsloth import FastLanguageModel

MODEL = "unsloth/Qwen3-8B-unsloth-bnb-4bit"  # Apache-2.0 weights, pre-quantized for T4
MAX_SEQ = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL, max_seq_length=MAX_SEQ, load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=32, lora_dropout=0.0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
)

In [ ]:
from datasets import load_dataset

# Upload training/dataset_policy-v1.jsonl via the Files panel, or mount Drive:
DATASET = "dataset_policy-v1.jsonl"  # <- path after upload
ds = load_dataset("json", data_files=DATASET, split="train").shuffle(seed=42)
print(ds)

In [ ]:
from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=ds,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=True,                 # T4 has no bf16
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        logging_steps=5,
        output_dir="qwen3-receptionist",
        dataset_text_field="",     # use chat template on the 'messages' field
    ),
)
trainer.train()

In [ ]:
# Push the LoRA adapter to a PRIVATE free Hugging Face repo
from huggingface_hub import login
login()  # paste a write token from https://huggingface.co/settings/tokens

REPO = "YOUR_HF_USERNAME/qwen3-8b-receptionist-lora-v1"
model.push_to_hub(REPO, private=True)
tokenizer.push_to_hub(REPO, private=True)

In [ ]:
# Optional: merged GGUF for local Ollama serving (CPU eval without any API)
model.save_pretrained_gguf("qwen3-8b-receptionist-gguf", tokenizer, quantization_method="q4_k_m")
# then:  ollama create receptionist -f Modelfile .   (see TRAINING.md)

## Before promoting

1. Serve the result (Ollama with the GGUF, or Groq/other OpenAI-compatible host for Qwen).
2. Point `.env` (`LLM_BASE_URL`, `LLM_MODEL`) at it.
3. Run `scripts/eval_battery.py` — **0 FAIL required**, including the Hindi/Hinglish asks.
4. Only then keep the `.env` override; otherwise revert two lines to Gemini. Version the adapter per policy version.